# 03 — Feature-Target Analysis

**Goal:** Understand the relationship between computed features and the label before training any model.

Run `build_features.py` and `create_labels.py` first:
```bash
cd /path/to/PriceDropOS
python ml/src/build_features.py
python ml/src/create_labels.py
```

Questions answered here:
- Is the class balance healthy enough to train on?
- Which features correlate most strongly with `label_drop_7d`?
- Are there features we can drop (near-zero correlation, too many NaNs)?
- What does the distribution of our label look like by retailer and category?
- Does the 5% threshold produce a reasonable split?

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

# Load features and labels generated by the scripts
features_path = '../data/features.csv'
labels_path   = '../data/labels.csv'

df_feat = pd.read_csv(features_path, parse_dates=['observed_at'])
df_lab  = pd.read_csv(labels_path,   parse_dates=['observed_at'])

# Keep only labelable rows
df_lab = df_lab[df_lab['is_labelable']].copy()

# Merge on observation id
df = df_feat.merge(df_lab[['id', 'label_drop_7d', 'price_drop_pct_7d']], on='id', how='inner')

print(f'Merged rows: {len(df):,}  |  Products: {df["product_id"].nunique()}')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/features.csv'

## 1. Class balance

In [ ]:
counts = df['label_drop_7d'].value_counts().sort_index()
print('Label distribution:')
print(counts.rename({0: 'No drop (0)', 1: 'Drop ≥5% (1)'}))
print(f'\nPositive rate: {df["label_drop_7d"].mean():.1%}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_xticklabels(['No drop', 'Drop ≥5%'], rotation=0)
axes[0].set_title('Class balance')

# Positive rate by retailer
pos_by_retailer = df.groupby('retailer')['label_drop_7d'].mean().sort_values()
pos_by_retailer.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Positive rate by retailer')
axes[1].set_xlabel('Fraction with price drop ≥5% in 7 days')
axes[1].axvline(0.5, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 2. Feature-target correlations

In [ ]:
numeric_features = [
    'price', 'price_avg_7d', 'price_avg_30d',
    'price_min_7d', 'price_min_30d', 'price_max_7d', 'price_max_30d',
    'price_std_7d', 'price_std_30d',
    'price_vs_avg_30d_pct', 'price_vs_min_30d_pct', 'price_vs_max_30d_pct',
    'price_change_prev', 'price_change_prev_pct',
    'num_drops_7d', 'num_drops_30d', 'days_since_last_drop',
    'observation_count', 'day_of_week', 'is_weekend', 'month',
]
# Keep only columns that exist in this dataset
numeric_features = [c for c in numeric_features if c in df.columns]

corr = df[numeric_features + ['label_drop_7d']].corr()['label_drop_7d'].drop('label_drop_7d').sort_values()

fig, ax = plt.subplots(figsize=(7, 8))
corr.plot(kind='barh', ax=ax, color=['coral' if v > 0 else 'steelblue' for v in corr])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson correlation with label_drop_7d')
ax.set_xlabel('Correlation coefficient')
plt.tight_layout()
plt.show()

print('\nTop positive correlators (features suggesting a drop is coming):')
print(corr.tail(5).to_string())
print('\nTop negative correlators (features suggesting no drop):')
print(corr.head(5).to_string())

## 3. NaN rates — which features are unreliable?

In [ ]:
nan_rates = df[numeric_features].isnull().mean().sort_values(ascending=False)
nan_rates = nan_rates[nan_rates > 0]

if nan_rates.empty:
    print('No NaN values in numeric features (dataset is dense).')
else:
    print('Features with NaN values (expected for cold-start observations):')
    print(nan_rates.round(3).to_string())
    print('\nNote: NaNs appear in early rows of each product where rolling windows')
    print('have fewer than min_periods observations. Use tree-based models that')
    print('handle NaN natively, or impute with -1 as a missing-indicator value.')

## 4. Feature distributions by label

In [ ]:
# Compare key feature distributions for label=0 vs label=1
key_features = [
    'price_vs_avg_30d_pct',
    'price_vs_min_30d_pct',
    'num_drops_30d',
    'price_std_30d',
]
key_features = [c for c in key_features if c in df.columns]

fig, axes = plt.subplots(1, len(key_features), figsize=(4 * len(key_features), 4))
if len(key_features) == 1:
    axes = [axes]

for ax, col in zip(axes, key_features):
    for label, color in [(0, 'steelblue'), (1, 'coral')]:
        sub = df[df['label_drop_7d'] == label][col].dropna()
        sub.hist(bins=30, alpha=0.6, ax=ax, color=color,
                 label=f'label={label} (n={len(sub)})')
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('Feature distributions: no-drop (blue) vs drop (coral)', y=1.02)
plt.tight_layout()
plt.show()

## 5. Threshold sensitivity

In [ ]:
if 'price_drop_pct_7d' in df.columns:
    thresholds = [0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15]
    rows = []
    for t in thresholds:
        pos = (df['price_drop_pct_7d'] >= t).sum()
        rows.append({'threshold': f'{t:.0%}',
                     'positive': pos,
                     'positive_rate': pos / len(df)})
    summary = pd.DataFrame(rows)
    print('Effect of threshold on positive class rate:')
    print(summary.to_string(index=False))
    print()
    print('Recommendation: pick the lowest threshold where the positive rate')
    print('is still interpretable as "meaningful savings" for shoppers.')
    print('A 5% drop on a $350 headphone = $17.50 — that matters.')
    print('A 5% drop on a $35 charger = $1.75 — might not matter to users.')

## 6. Key takeaways

In [ ]:
print('=== KEY TAKEAWAYS ===')
pos_rate = df['label_drop_7d'].mean()
print(f'Positive class rate         : {pos_rate:.1%}')
if pos_rate < 0.10:
    print('  → Class is imbalanced. Use class_weight="balanced" or SMOTE in Phase 3.')
elif pos_rate > 0.50:
    print('  → Positive rate is high — the threshold may be too low.')
else:
    print('  → Reasonable balance for binary classification.')

top_corr = corr.abs().sort_values(ascending=False).head(3)
print(f'\nTop 3 most predictive features:')
for feat, val in top_corr.items():
    print(f'  {feat:<35} |r| = {val:.3f}')

high_nan = nan_rates[nan_rates > 0.30] if not nan_rates.empty else pd.Series(dtype=float)
print(f'\nFeatures with >30% NaN (consider dropping or imputing):')
if high_nan.empty:
    print('  None.')
else:
    for feat, rate in high_nan.items():
        print(f'  {feat}: {rate:.0%}')